# Unitary Manifold — Notebook 01: Quickstart

This notebook demonstrates the core field evolution pipeline:
- Initialise a flat Minkowski background with small perturbations
- Run the Walker–Pearson explicit-Euler integrator
- Monitor constraint violations and the conserved information current
- Visualise scalar field φ and Ricci scalar R over time

**Repository:** https://github.com/wuzbak/Unitary-Manifold-

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import matplotlib.pyplot as plt

from src.core.evolution import FieldState, run_evolution, constraint_monitor, information_current
from src.core.metric import compute_curvature

plt.rcParams['figure.dpi'] = 100

## 1 · Initialise flat background

In [ ]:
N, dx = 64, 0.1
state = FieldState.flat(N=N, dx=dx, rng=np.random.default_rng(42))

print(f"Grid points : {N}")
print(f"Grid spacing: dx = {dx}")
print(f"Initial φ   : mean={state.phi.mean():.6f}, std={state.phi.std():.2e}")
print(f"Initial |B| : mean={np.linalg.norm(state.B, axis=1).mean():.2e}")

## 2 · Run field evolution

In [ ]:
dt, steps = 1e-3, 300

history = run_evolution(state, dt=dt, steps=steps)

print(f"Steps evolved : {steps}")
print(f"Final time    : t = {history[-1].t:.3f}")
print(f"Final φ range : [{history[-1].phi.min():.6f}, {history[-1].phi.max():.6f}]")

## 3 · Visualise scalar field φ

In [ ]:
x = np.arange(N) * dx

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, idx, label in zip(axes,
                           [0, steps // 2, steps],
                           ['t = 0', f't = {history[steps//2].t:.3f}', f't = {history[steps].t:.3f}']):
    ax.plot(x, history[idx].phi)
    ax.set_title(label)
    ax.set_xlabel('x')
    ax.set_ylabel('φ')

plt.suptitle('Entanglement-capacity scalar φ at three time slices')
plt.tight_layout()
plt.show()

## 4 · Ricci scalar over time

In [ ]:
times = []
R_means = []
R_maxes = []

for s in history[::10]:   # sample every 10 steps
    _, _, _, R = compute_curvature(s.g, s.B, s.phi, s.dx)
    times.append(s.t)
    R_means.append(float(R.mean()))
    R_maxes.append(float(np.abs(R).max()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, R_means, label='⟨R⟩')
ax.plot(times, R_maxes, label='max|R|', linestyle='--')
ax.set_xlabel('time')
ax.set_ylabel('Ricci scalar R')
ax.set_title('Scalar curvature evolution')
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Constraint monitor

In [ ]:
final = history[-1]
_, _, Ricci_f, R_f = compute_curvature(final.g, final.B, final.phi, final.dx)
constraints = constraint_monitor(Ricci_f, R_f, final.B, final.phi)

print("Constraint violations at final time:")
for k, v in constraints.items():
    print(f"  {k:25s} = {v:.4e}")

## 6 · Conserved information current J^μ_inf

In [ ]:
J = information_current(final.g, final.phi, final.dx)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, J[:, 0], label='J⁰ (time component)')
ax.plot(x, J[:, 1], label='J¹ (spatial component)', linestyle='--')
ax.set_xlabel('x')
ax.set_ylabel('J^μ_inf')
ax.set_title('Information current at final time')
ax.legend()
plt.tight_layout()
plt.show()

# Check conservation ∇_μ J^μ ≈ 0
div_J = np.gradient(J[:, 1], dx, edge_order=2)
print(f"max |∇·J| = {np.abs(div_J).max():.4e}  (should be small for flat background)")